# 画像と音声のAIを使ってみよう（2026年版）

このノートブックで実行するのは **3つのセルだけ** です。3つ目のセルで出るボタンを押すと、新しいタブに **アプリ** が開きます。あとはアプリの中で試します。

| 番号 | アプリのページ | 使うもの |
|---|---|---|
| 1 | 画像とピクセル・画像分類（体験1） | 無料 |
| 2 | 手と体の検出（体験2） | 無料 |
| 3 | LLMで画像を読み取る | OpenAI の API（`gpt-6-luna`） |
| 4 | 音の波形とスペクトログラム（体験3） | 無料 |
| 5 | 文字起こしの実験 | OpenAI の API（`gpt-transcribe`） |
| 6 | 声で話すChatbot | OpenAI の API（3つ） |
| 7 | 意見メモ（第3部） | 無料（「AIに聞く」だけ OpenAI の API（`gpt-6-luna`）） |

### 気をつけること
- 3・5・6 では写真や声が、7 の「AIに聞く」ではメモの文字が、OpenAI のサーバーに送られます。**自分や友達の顔、名前・住所などの個人情報は送らない**でください。
- API を使うたびに、その1回の料金と、このアプリで使った料金の合計がページに出ます。
- カメラやマイクの許可を聞かれたら「許可」を押します。
- APIキーは、この Colab の中だけに置かれます。アプリのページ（ブラウザ）には渡しません。

## 0-1 準備
▶ を押します。1分ほどかかります。

In [ ]:
# @title 準備（中身は読まなくてよい）
!pip install -q -U openai
import base64, inspect, io, json, os, re, subprocess, tempfile, threading, time, traceback, urllib.request, zlib
from datetime import datetime, timedelta, timezone
import openai
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly, spectrogram
from matplotlib.figure import Figure
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, ImageOps
from IPython.display import display, HTML

MODEL_TEXT = "gpt-6-luna"        # 写真を読む・会話する
MODEL_STT = "gpt-transcribe"     # 文字起こし
MODEL_TTS = "gpt-4o-mini-tts"    # 音声合成
# 料金（2026年10月時点の公式の料金表。https://developers.openai.com/api/docs/pricing）
YEN = 150                                     # 1ドル＝150円で計算
PRICE_IN, PRICE_CACHED, PRICE_OUT = 0.10, 0.01, 0.50   # gpt-6-luna（100万トークンあたりのドル。入力・前と同じ部分の入力・出力）
PRICE_STT = 0.0045                            # gpt-transcribe（1分あたりのドル）
PRICE_TTS_IN, PRICE_TTS_OUT = 0.60, 12.00     # gpt-4o-mini-tts（100万トークンあたりのドル。文字の入力・声の出力）
SR = 16000                       # 文字起こしの声は、AIに送る形と同じ 1秒に 16,000 回・1チャンネル
RATES = [16000, 8000, 4000, 3000, 2000]
VOICES = ["marin", "cedar", "alloy", "ash", "ballad", "coral", "echo", "fable", "nova", "onyx", "sage", "shimmer", "verse"]

# アプリのページ（体験1〜3と、API を使う4ページ）
PAGES = json.loads(zlib.decompress(base64.b64decode("""
...
""".replace("\n", ""))).decode("utf-8"))
ROUTES = {"/": "menu.html", "/1": "taiken1", "/2": "taiken2", "/3": "read_image.html", "/4": "taiken3",
          "/5": "transcribe.html", "/6": "voice_chat.html", "/7": "memo.html", "/app.css": "app.css", "/app.js": "app.js"}


# ---------- 部品 ----------
def from_data_url(u):
    return base64.b64decode(u.split(",", 1)[1])


def to_data_url(b, mime):
    return f"data:{mime};base64," + base64.b64encode(b).decode()


def to_wav(data_url, path, rate):
    """ブラウザで録音した声を、1秒に rate 回測った wav にする"""
    with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as f:
        f.write(from_data_url(data_url))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f.name, "-ac", "1", "-ar", str(rate), path], check=True)
    os.remove(f.name)
    r, x = wavfile.read(path)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if x.dtype == np.int16:
        x = x / 32768.0
    return r, x.astype(np.float32)


def wav_bytes(rate, x):
    peak = float(np.max(np.abs(x))) if len(x) else 0
    if peak > 1:
        x = x / peak
    buf = io.BytesIO()
    wavfile.write(buf, rate, (x * 32767).astype(np.int16))
    return buf.getvalue()


def measure(x, src, rate):
    """1秒に rate 回だけ測った音（rate の半分より上の高さは消える）"""
    return x if rate >= src else resample_poly(x, rate, src).astype(np.float32)


def back_to(y, rate, src, n):
    """聞いたり API に送ったりするために、元の測る回数の形に戻す"""
    return y if rate >= src else resample_poly(y, src, rate).astype(np.float32)[:n]


def add_noise(x, percent, seed=0):
    """声の大きさの percent % の大きさのザーッという音を足す"""
    if percent <= 0:
        return x
    rms = float(np.sqrt(np.mean(x ** 2))) or 1e-3
    return x + np.random.default_rng(seed).normal(0, rms * percent / 100, size=x.shape).astype(np.float32)


CMAP = LinearSegmentedColormap.from_list("note", ["#ffffff", "#ffe76a", "#f57c00", "#782800", "#1e1e1e"])


def spec_png(x, rate, cut=None, max_hz=8000):
    """スペクトログラム（横＝時間、たて＝音の高さ、濃さ＝強さ）の画像"""
    n = 2 ** int(round(np.log2(rate * 0.025)))
    f, t, s = spectrogram(x, fs=rate, window="hann", nperseg=n, noverlap=n * 3 // 4)
    db = 10 * np.log10(s + 1e-12)
    fig = Figure(figsize=(6, 2.2), dpi=110)
    ax = fig.add_subplot()
    ax.pcolormesh(t, f, db, cmap=CMAP, vmin=db.max() - 70, vmax=db.max(), shading="auto")
    ax.set_ylim(0, max_hz)
    ax.set_xlabel("s")
    ax.set_ylabel("Hz")
    if cut and cut / 2 < max_hz:
        ax.axhspan(cut / 2, max_hz, color="#e6e6e1", alpha=0.85)
        ax.axhline(cut / 2, color="#d35400", linewidth=2)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    return to_data_url(buf.getvalue(), "image/png")


def mel_png(x, rate, n_mels=80):
    """多くの音声認識AIが読む形：0.025秒ずつ、0.01秒ずらして、低い音〜高い音を80段（メル）に分けたマスの画像"""
    n, hop, nfft = int(rate * 0.025), int(rate * 0.01), 512
    x = np.pad(x, (0, max(0, n - len(x))))
    frames = 1 + (len(x) - n) // hop
    power = np.abs(np.fft.rfft(x[np.arange(n)[None, :] + hop * np.arange(frames)[:, None]] * np.hanning(n), nfft)) ** 2
    to_mel, from_mel = (lambda f: 2595 * np.log10(1 + f / 700)), (lambda m: 700 * (10 ** (m / 2595) - 1))
    edges = np.floor((nfft + 1) * from_mel(np.linspace(0, to_mel(rate / 2), n_mels + 2)) / rate).astype(int)
    fb = np.zeros((n_mels, nfft // 2 + 1))
    for i in range(n_mels):   # 三角形の重みで、となりの段と少し重ねて足す
        l, c, r = edges[i], edges[i + 1], max(edges[i + 2], edges[i + 1] + 1)
        fb[i, l:c] = (np.arange(l, c) - l) / max(1, c - l)
        fb[i, c:r] = (r - np.arange(c, r)) / max(1, r - c)
    m = np.log10(np.maximum(power @ fb.T, 1e-10))
    m = np.maximum(m, m.max() - 8)
    m = (m - m.min()) / (m.max() - m.min() + 1e-9)
    rgb = (CMAP(m.T[::-1])[:, :, :3] * 255).astype(np.uint8)   # 上が高い音
    buf = io.BytesIO()
    Image.fromarray(rgb).resize((frames * 2, n_mels * 2), Image.NEAREST).save(buf, format="PNG")
    return to_data_url(buf.getvalue(), "image/png"), frames


def transcribe(path, prompt="", keywords=None, languages=None):
    extra = {}
    if keywords:
        extra["keywords"] = keywords
    if languages:
        extra["languages"] = languages
    with open(path, "rb") as f:
        args = {"model": MODEL_STT, "file": f}
        if prompt:
            args["prompt"] = prompt
        if extra:
            args["extra_body"] = extra
        return client.audio.transcriptions.create(**args).text


def transcribe_stream(key, path, prompt="", keywords=None, languages=None):
    """文字起こし。できたところから少しずつ届く（transcript.text.delta）。届いたかけらを1つずつ渡し、最後に全文を返す"""
    args = {"model": MODEL_STT, "stream": True}
    if prompt:
        args["prompt"] = prompt
    extra = {k: v for k, v in (("keywords", keywords), ("languages", languages)) if v}
    if extra:
        args["extra_body"] = extra
    text = ""
    yield {"type": "start", "key": key}
    with open(path, "rb") as f:
        for ev in client.audio.transcriptions.create(file=f, **args):
            if ev.type == "transcript.text.delta":
                text += ev.delta
                yield {"type": "delta", "key": key, "text": ev.delta}
            elif ev.type == "transcript.text.done":
                text = ev.text
    return text


def speak(args):
    """文字を声にする。stream_format="sse" で頼むと声が少しずつ届き、最後の知らせ（speech.audio.done）に使ったトークン数が付く。
    かけらが届くたびに知らせ、最後に（声, トークン数）を返す"""
    try:
        audio, usage, n = b"", None, 0
        with client.audio.speech.with_streaming_response.create(**args, stream_format="sse") as r:
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                data = line[5:].strip()
                if data == "[DONE]":
                    break
                ev = json.loads(data)
                if ev.get("type") == "speech.audio.delta":
                    audio += base64.b64decode(ev["audio"])
                    n += 1
                    yield {"type": "audio_chunk", "n": n, "kb": len(audio) / 1024}
                elif ev.get("type") == "speech.audio.done":
                    usage = ev.get("usage")
        if audio:
            return audio, usage
    except (TypeError, openai.BadRequestError):
        pass
    return client.audio.speech.create(**args).content, None   # sse で頼めないときは、ふつうに頼む（料金は分からない）


# ---------- 料金（円） ----------
SPENT = {"yen": 0.0}   # このアプリで使った料金の合計（Colab のランタイムが続くあいだ数える）


def spend(yen):
    SPENT["yen"] += yen
    return yen


def stt_yen(seconds, times=1):
    return seconds / 60 * PRICE_STT * YEN * times


def luna_yen(usage):
    cached = getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0
    return ((usage.input_tokens - cached) * PRICE_IN + cached * PRICE_CACHED + usage.output_tokens * PRICE_OUT) / 1_000_000 * YEN


def tts_yen(usage):
    return (usage.get("input_tokens", 0) * PRICE_TTS_IN + usage.get("output_tokens", 0) * PRICE_TTS_OUT) / 1_000_000 * YEN


# ---------- うまく動かないときのための記録と見立て（0-3 と 0-4 で見る） ----------
LOG = {"requests": [], "errors": [], "port": None, "started": None}
PAGE_NAMES = {"/": "メニュー", "/1": "1 画像分類", "/2": "2 手と体", "/3": "3 画像を読む", "/4": "4 音を見る",
              "/5": "5 文字起こし", "/6": "6 声で話す", "/7": "7 意見メモ", "/ping": "Colab の中からの確認",
              "status": "メニュー", "diag": "診断", "ask_image": "3 画像を読む", "voice": "5 録音", "exp_a": "5 実験A",
              "exp_b": "5 実験B", "chat": "6 声で話す", "next": "7 AIに聞く"}


def now():
    return datetime.now(timezone(timedelta(hours=9))).strftime("%H:%M:%S")   # 日本の時刻


def note_request(method, path, code, ms, self_check):
    if path in ("/app.css", "/app.js"):
        return
    name = PAGE_NAMES.get(path) or PAGE_NAMES.get(path.rsplit("/", 1)[-1]) or path
    LOG["requests"].append({"time": now(), "page": name, "code": code, "ms": ms, "self": self_check})
    del LOG["requests"][:-40]


def explain(e):
    """エラーを「番号・何が起きたか・原因の見立て・直し方」にする"""
    status = getattr(e, "status_code", None)
    body = getattr(e, "body", None)
    code = (body.get("code") if isinstance(body, dict) else None) or getattr(e, "code", None)
    def out(title, cause="", fix="", num=status):
        return {"code": num, "title": title, "cause": cause, "fix": fix}
    if isinstance(e, NameError) and "client" in str(e):
        return out("APIキーがまだ設定されていません", "0-2 を実行していない（ランタイムが切れると消える）", "Colab の 0-2 を実行する")
    if isinstance(e, ValueError):
        return out(str(e))
    if isinstance(e, openai.APITimeoutError):
        return out("OpenAI からの返事が時間内に来ませんでした", "OpenAI が混んでいる・送るものが大きい", "少し待って、もう一度押す")
    if isinstance(e, openai.APIConnectionError):
        return out("Colab から OpenAI につながりませんでした", "ネットワークの一時的な不調", "少し待って、もう一度押す")
    if status == 429 and code == "insufficient_quota":
        return out("使える金額の上限に達しました", "このキーのプロジェクトの残高・上限を使い切った", "講師に伝える")
    table = {
        400: ("送った中身を OpenAI が受け付けませんでした", "設定の値や、モデルとの組み合わせが合っていない", "「くわしく」を講師に見せる"),
        401: ("APIキーが正しくありません", "キーの貼り間違い（前後の空白など）・無効になったキー", "Colab の 0-2 をやり直す。直らなければ講師に"),
        403: ("このキーでは、このモデルや機能を使えません", "キーのプロジェクトで、このモデルが許可されていない", "講師に伝える"),
        404: ("モデルが見つかりません", "モデルの名前が違う・このキーでは使えないモデル", "講師に伝える"),
        429: ("短い時間に頼みすぎました", "1分あたりの回数やトークンの上限をこえた", "30秒ほど待って、もう一度押す"),
    }
    if status in table:
        return out(*table[status])
    if status and status >= 500:
        return out("OpenAI の中でエラーが起きました", "OpenAI 側の一時的な不調", "少し待って、もう一度押す")
    return out(f"思わぬエラー（{type(e).__name__}）", "", "「くわしく」を講師に見せる。Colab の 0-4 で記録を見られる")


def note_error(where, e):
    info = explain(e)
    info["detail"] = "".join(traceback.format_exception_only(type(e), e)).strip()[:800]
    LOG["errors"].append({"time": now(), "page": PAGE_NAMES.get(where, where), **info})
    del LOG["errors"][:-10]
    return info


def check_server():
    port = LOG["port"]
    if not port:
        return False, "まだ起動していません。0-3 を実行する"
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{port}/ping", headers={"X-Self-Check": "1"})
        with urllib.request.urlopen(req, timeout=5) as r:
            if r.status == 200:
                return True, f"動いています（ポート {port}。{LOG['started']} に起動）"
            return False, f"Colab の中から開くと {r.status} が返りました。0-3 を実行し直す"
    except Exception as e:
        return False, f"Colab の中から開けません（{type(e).__name__}）。0-3 を実行し直す"


def check_key():
    key = os.environ.get("OPENAI_API_KEY", "")
    if "client" not in globals() or not key:
        return False, "まだ設定されていません。0-2 を実行する"
    if key != key.strip() or not key.startswith("sk-"):
        return False, "形がおかしいかもしれません（前後の空白・貼り間違い）。0-2 をやり直す"
    return True, "設定済み"


def check_models():
    """モデルが使えるかを、モデルの情報を読むだけで確かめる（料金はかからない）"""
    out = []
    for m in (MODEL_TEXT, MODEL_STT, MODEL_TTS):
        try:
            client.models.retrieve(m)
            out.append({"model": m, "ok": True})
        except Exception as e:
            out.append({"model": m, "ok": False, **explain(e)})
    return out


def diag_data(models=False):
    server, key = check_server(), check_key()
    browser = [r for r in LOG["requests"] if not r["self"]]
    d = {"server": server, "key": key, "browser": browser[-10:], "browser_count": len(browser), "errors": LOG["errors"][-5:],
         "models": check_models() if models and key[0] else []}
    if not server[0]:
        d["guess"] = "サーバーが止まっています。0-3 を実行し直す。直らなければ「ランタイム → セッションを再起動」して 0-1 から"
    elif not browser:
        d["guess"] = ("ブラウザからアプリに1回も届いていません。開いたときに 403 などで止まっている見込みです。"
                      "Colab を開いているのと同じブラウザで開く・ブラウザでサードパーティ Cookie を止めていないか確かめる・0-3b（セルの中で開く）を試す")
    elif not key[0]:
        d["guess"] = key[1]
    elif any(not m["ok"] for m in d["models"]):
        m = next(m for m in d["models"] if not m["ok"])
        d["guess"] = f"{m['model']}：{m['title']}。{m['fix']}"
    elif d["errors"]:
        e = d["errors"][-1]
        d["guess"] = f"最後のエラー（{e['time']}・{e['page']}）：{e['title']}。{e['fix']}"
    else:
        d["guess"] = "見つかった問題はありません"
    return d


def esc(t):
    return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


CSS = "font:16px sans-serif;line-height:1.7;"


def diagnose():
    """0-4 で見せる診断の結果（HTML）"""
    d = diag_data(models=True)
    mark = lambda ok: "✅" if ok else "❌"
    rows = [f"<tr><td>{mark(d['server'][0])}</td><td><b>サーバー</b></td><td>{esc(d['server'][1])}</td></tr>",
            f"<tr><td>{mark(d['key'][0])}</td><td><b>APIキー</b></td><td>{esc(d['key'][1])}</td></tr>"]
    for m in d["models"]:
        rows.append(f"<tr><td>{mark(m['ok'])}</td><td><b>{esc(m['model'])}</b></td><td>"
                    + ("使えます" if m["ok"] else f"{m['code'] or ''} {esc(m['title'])}。{esc(m['fix'])}") + "</td></tr>")
    n = d["browser_count"]
    rows.append(f"<tr><td>{mark(n > 0)}</td><td><b>ブラウザから</b></td><td>"
                + (f"{n} 回届いています（最後は {d['browser'][-1]['time']}・{esc(d['browser'][-1]['page'])}・{d['browser'][-1]['code']}）"
                   if n else "まだ1回も届いていません") + "</td></tr>")
    html = (f"<div style='{CSS}'><p style='font-size:20px;font-weight:900;margin:6px 0;'>いちばん怪しいところ："
            f"<span style='background:#ffe76a;'>{esc(d['guess'])}</span></p>"
            f"<table style='border-collapse:collapse;'>{''.join(rows)}</table>")
    if d["browser"]:
        html += "<p style='margin-top:10px;font-weight:900;'>ブラウザから届いたアクセス（新しい順）</p><table style='border-collapse:collapse;'>"
        for r in reversed(d["browser"]):
            html += f"<tr><td>{r['time']}</td><td style='padding:0 12px;'>{esc(r['page'])}</td><td>{r['code']}</td><td style='padding-left:12px;'>{r['ms']} ms</td></tr>"
        html += "</table>"
    if d["errors"]:
        html += "<p style='margin-top:10px;font-weight:900;'>最近のエラー（新しい順）</p>"
        for e in reversed(d["errors"]):
            html += (f"<div style='margin:6px 0;padding:8px 12px;border-left:4px solid #c0392b;background:#fdf0ee;'>"
                     f"<b>{e['time']}・{esc(e['page'])}・{e['code'] or ''} {esc(e['title'])}</b><br>原因の見立て：{esc(e['cause']) or '—'}"
                     f"<br>直し方：{esc(e['fix']) or '—'}<br><code style='font-size:13px;'>{esc(e['detail'])}</code></div>")
    return html + "</div>"


def start_panel(url, port):
    """0-3 の結果：サーバーとAPIキーの状態、開くボタン、開かないときの表"""
    (s_ok, s_msg), (k_ok, k_msg) = check_server(), check_key()
    mark = lambda ok: "✅" if ok else "❌"
    btn = (f'<a href="{url}" target="_blank" style="display:inline-block;margin:12px 0;padding:18px 40px;border-radius:999px;'
           'background:#f57c00;color:#fff;font:900 24px sans-serif;text-decoration:none;">▶ アプリを開く（新しいタブ）</a>') if s_ok else ""
    rows = [("403・「アクセスできません」", "ブラウザが Colab の接続に使う Cookie（サードパーティ Cookie）を止めている。Colab と別のブラウザ・シークレットウィンドウで開いた",
             "Colab を開いているのと同じブラウザで開く。ブラウザの設定で Cookie を許可する。0-3b を試す"),
            ("404", "開いたURLが古い", "このセルの新しいボタンから開く"),
            ("500・502・503・504、ずっと読み込み中", "サーバーかランタイムが止まった", "このセルを実行し直す。直らなければ 0-4"),
            ("アプリの中のエラー", "ページに、番号・原因の見立て・直し方が出る", "直し方のとおりにする。分からなければ 0-4")]
    table = "".join(f"<tr><td style='padding:4px 10px;font-weight:900;'>{a}</td><td style='padding:4px 10px;'>{b}</td><td style='padding:4px 10px;'>{c}</td></tr>" for a, b, c in rows)
    return (f"<div style='{CSS}'>{mark(s_ok)} サーバー：{esc(s_msg)}<br>{mark(k_ok)} APIキー：{esc(k_msg)}<br>{btn}"
            f"<p>ボタンで開かないときは、このURLを新しいタブで開く：<br><code>{esc(url)}</code></p>"
            f"<details><summary style='cursor:pointer;font-weight:900;'>開かないとき（見えたもの → 原因の見立て → 直し方）</summary>"
            f"<table style='border-collapse:collapse;margin-top:6px;'>{table}</table></details></div>")


def api_diag(d):
    return diag_data(models=bool(d.get("models")))


# ---------- アプリから呼ばれる処理（ここで OpenAI の API を使う） ----------
STATE = {"voice": None, "orig_text": None}


def api_status(d):
    return {"key": bool(os.environ.get("OPENAI_API_KEY"))}


# JSON で受け取るときの形（物の名前・数・位置）。位置は、画像の幅と高さをそれぞれ 1 としたときの [左, 上, 右, 下]
OBJ_SCHEMA = {
    "type": "object",
    "properties": {"objects": {"type": "array", "items": {
        "type": "object",
        "properties": {"name": {"type": "string", "description": "物の名前（日本語）"},
                       "count": {"type": "integer", "description": "その物の数"},
                       "box": {"type": "array", "items": {"type": "number"},
                               "description": "位置。画像の幅と高さを 1 とした [左, 上, 右, 下]"}},
        "required": ["name", "count", "box"], "additionalProperties": False}}},
    "required": ["objects"], "additionalProperties": False,
}


def api_ask_image(d):
    ms = lambda t: round((time.time() - t) * 1000)
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(from_data_url(d["image"])))).convert("RGB")
    detail = "high" if d.get("detail") == "high" else "low"
    mode = "json" if d.get("mode") == "json" else "text"
    # AIに届く大きさ（2026年10月時点の公式ドキュメント）：low は 512 以内、high は 2048 以内で 32×32 のマスが 2,500 まで。
    # 届く形が見えるように、こちらでその大きさに縮めてから送る
    s = min(1.0, (512 if detail == "low" else 2048) / max(img.size))
    if detail == "high":
        s = min(s, (2500 * 32 * 32 / (img.width * img.height)) ** 0.5)
    if s < 1:
        img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    url = to_data_url(buf.getvalue(), "image/jpeg")
    request = {"model": MODEL_TEXT, "reasoning": {"effort": "low"}, "input": [{"role": "user", "content": [
        {"type": "input_text", "text": d["question"][:1000]},
        {"type": "input_image", "image_url": url, "detail": detail},
    ]}]}
    if mode == "json":   # 決めた形の JSON で返してもらう（作品のプログラムで使える形）
        request["text"] = {"format": {"type": "json_schema", "name": "objects", "strict": True, "schema": OBJ_SCHEMA}}
    shown = json.loads(json.dumps(request))   # 見せる用：写真の文字は先頭だけ
    shown["input"][0]["content"][1]["image_url"] = f"{url[:40]}…（写真を文字にしたもの。全部で {len(url):,} 文字）"
    yield {"type": "sent", "detail": detail, "mode": mode, "sent": [img.width, img.height],
           "patches": [-(-img.width // 32), -(-img.height // 32)], "image": url, "request": shown, "chars": len(url)}
    t, text, first, resp = time.time(), "", None, None
    for ev in client.responses.create(**request, stream=True):
        if ev.type == "response.output_text.delta":
            first = first if first is not None else ms(t)
            text += ev.delta
            yield {"type": "delta", "text": ev.delta}
        elif ev.type == "response.completed":
            resp = ev.response
    u = resp.usage
    reasoning = getattr(getattr(u, "output_tokens_details", None), "reasoning_tokens", 0) or 0
    cached = getattr(getattr(u, "input_tokens_details", None), "cached_tokens", 0) or 0
    data = None
    if mode == "json":
        try:
            data = json.loads(text)
        except ValueError:
            data = None
    yield {"type": "done", "text": text, "data": data, "ms": ms(t), "first_ms": first,
           "input_tokens": u.input_tokens, "cached_tokens": cached, "output_tokens": u.output_tokens, "reasoning_tokens": reasoning,
           "yen_in": ((u.input_tokens - cached) * PRICE_IN + cached * PRICE_CACHED) / 1_000_000 * YEN,
           "yen_out": u.output_tokens * PRICE_OUT / 1_000_000 * YEN, "yen": spend(luna_yen(u)),
           "response": {"model": resp.model, "status": resp.status, "output_text": text, "usage": u.model_dump()}}


def api_voice(d):
    rate, x = to_wav(d["audio"], "voice.wav", SR)
    STATE.update(voice=(rate, x), orig_text=None)
    mel, frames = mel_png(x, rate)
    return {"seconds": len(x) / rate, "spec": spec_png(x, rate), "rate": rate, "kb": os.path.getsize("voice.wav") / 1024,
            "mel": mel, "frames": frames}


def need_voice():
    if STATE["voice"] is None:
        raise ValueError("先に録音してください")
    return STATE["voice"]


def orig_text(key):
    """元の声の文字起こし（同じ声なら1回だけ API を使い、2回目からは前の結果を使う）"""
    if STATE["orig_text"] is None:
        STATE["orig_text"] = yield from transcribe_stream(key, "voice.wav")
        return STATE["orig_text"], 1
    yield {"type": "text", "key": key, "text": STATE["orig_text"]}
    return STATE["orig_text"], 0


def api_exp_a(d):
    rate, x = need_voice()
    noise = max(0, min(300, int(d.get("noise", 0))))
    target = int(d.get("rate", 16000))
    if target not in RATES:
        raise ValueError("サンプリングレートの値が正しくありません")
    changed = add_noise(back_to(measure(x, rate, target), target, rate, len(x)), noise)
    data = wav_bytes(rate, changed)
    with open("voice_changed.wav", "wb") as f:
        f.write(data)
    yield {"type": "info", "orig_spec": spec_png(x, rate), "changed_spec": spec_png(changed, rate, cut=target),
           "changed_audio": to_data_url(data, "audio/wav")}
    text0, calls = yield from orig_text("A0")
    text1 = yield from transcribe_stream("A1", "voice_changed.wav")
    yield {"type": "result", "orig_text": text0, "changed_text": text1, "calls": calls + 1,
           "yen": spend(stt_yen(len(x) / rate, calls + 1))}


def api_exp_b(d):
    rate, x = need_voice()
    kw = [k for k in d.get("keywords", []) if isinstance(k, str) and k.strip()][:20]
    langs = [l for l in d.get("languages", []) if l in ("ja", "en")]
    plain, calls = yield from orig_text("B0")
    hinted = yield from transcribe_stream("B1", "voice.wav", prompt=d.get("prompt", "")[:300], keywords=kw, languages=langs)
    yield {"type": "result", "plain": plain, "hinted": hinted, "calls": calls + 1,
           "yen": spend(stt_yen(len(x) / rate, calls + 1))}


def api_chat(d):
    """3つのAIのリレー：声 → 文字（gpt-transcribe）→ 返事（gpt-6-luna）→ 声（gpt-4o-mini-tts）。区切りごとに知らせる"""
    ms = lambda t: round((time.time() - t) * 1000)
    rate, x = to_wav(d["audio"], "talk.wav", SR)
    t = time.time()
    user_text = transcribe("talk.wav", languages=["ja"]).strip()
    stt = spend(stt_yen(len(x) / rate))
    if not user_text:
        raise ValueError(f"声が聞き取れませんでした。もう一度話してください（文字起こしの料金 約 {stt:.3f} 円）")
    yield {"type": "heard", "text": user_text, "ms": ms(t), "yen": stt, "seconds": len(x) / rate,
           "kb": os.path.getsize("talk.wav") / 1024, "chars": len(user_text)}
    history = [{"role": m["role"], "content": str(m["content"])} for m in d.get("history", [])
               if isinstance(m, dict) and m.get("role") in ("user", "assistant")][-20:]
    messages = [{"role": "system", "content": d.get("system", "")}] + history + [{"role": "user", "content": user_text}]
    yield {"type": "sent", "request": {"model": MODEL_TEXT, "reasoning": {"effort": "low"}, "input": messages}}
    t, reply, first, usage = time.time(), "", None, None
    for ev in client.responses.create(model=MODEL_TEXT, reasoning={"effort": "low"}, input=messages, stream=True):
        if ev.type == "response.output_text.delta":
            first = first if first is not None else ms(t)
            reply += ev.delta
            yield {"type": "delta", "text": ev.delta}
        elif ev.type == "response.completed":
            usage = ev.response.usage
    cached = getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0 if usage else 0
    yield {"type": "reply", "text": reply, "ms": ms(t), "first_ms": first, "yen": spend(luna_yen(usage)) if usage else 0,
           "input_tokens": usage.input_tokens if usage else 0, "output_tokens": usage.output_tokens if usage else 0,
           "cached_tokens": cached, "chars": len(reply)}
    args = {"model": MODEL_TTS, "voice": d.get("voice") if d.get("voice") in VOICES else "marin", "input": reply,
            "speed": max(0.5, min(2.0, float(d.get("speed", 1.0)))), "response_format": "mp3"}
    if d.get("instructions"):
        args["instructions"] = d["instructions"]
    t = time.time()
    yield {"type": "tts_start"}
    audio, tts_usage = yield from speak(args)
    yield {"type": "audio", "audio": to_data_url(audio, "audio/mpeg"), "ms": ms(t), "kb": len(audio) / 1024,
           "yen": spend(tts_yen(tts_usage)) if tts_usage else None, "tts_usage": tts_usage}


NEXT = (
    "あなたは、中学生・高校生の作品づくりを手伝う先輩です。"
    "作品を見せたときにグループの人から出た意見のメモ（使ってどう思ったか／欲しい機能／開発の仕方）を読み、"
    "次の1週間でやることの候補を3つ出してください。"
    "意見の中から、作品をよくするのに大事なものを選び、1つの候補は何をするかが分かる短い1文にしてください。"
    "否定はせず、中学生にも分かる日本語で、番号を付けて候補だけを3行で書いてください。"
)


def api_next(d):
    memo = (f"作品名：{d.get('title', '')[:100]}\n使ってどう思ったか：{d.get('feel', '')[:1000]}\n"
            f"欲しい機能：{d.get('want', '')[:1000]}\n開発の仕方：{d.get('dev', '')[:1000]}")
    resp = client.responses.create(model=MODEL_TEXT, reasoning={"effort": "low"},
                                   input=[{"role": "system", "content": NEXT}, {"role": "user", "content": memo}])
    lines = [re.sub(r"^\s*([0-9０-９]+[.)．、）]|[-・*])\s*", "", l).strip() for l in resp.output_text.splitlines()]
    return {"ideas": [l for l in lines if l][:3] or [resp.output_text], "yen": spend(luna_yen(resp.usage)),
            "input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}


API = {"status": api_status, "ask_image": api_ask_image, "voice": api_voice, "exp_a": api_exp_a,
       "exp_b": api_exp_b, "chat": api_chat, "next": api_next, "diag": api_diag}


class AppHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def note(self, code, t0):
        note_request(self.command, self.path.split("?")[0].rstrip("/") or "/", code, round((time.time() - t0) * 1000),
                     self.headers.get("X-Self-Check") == "1")

    def reply(self, code, body, ctype):
        self.send_response(code)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        t0 = time.time()
        path = self.path.split("?")[0].rstrip("/") or "/"
        if path == "/ping":
            self.reply(200, b'{"ok": true}', "application/json")
            return self.note(200, t0)
        name = ROUTES.get(path)
        if name is None:
            self.reply(404, "ページがありません".encode(), "text/plain; charset=utf-8")
            return self.note(404, t0)
        ctype = {"css": "text/css", "js": "application/javascript"}.get(name.rsplit(".", 1)[-1], "text/html")
        self.reply(200, PAGES[name].encode("utf-8"), ctype + "; charset=utf-8")
        self.note(200, t0)

    def do_POST(self):
        t0 = time.time()
        name = self.path.split("?")[0].rstrip("/").rsplit("/", 1)[-1]
        fn = API.get(name) if "/api/" in self.path else None
        if fn is None:
            self.reply(404, b"{}", "application/json")
            return self.note(404, t0)
        try:
            data = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
            out, code = fn(data), 200
        except Exception as e:
            info = note_error(name, e)
            out, code = {"error": info}, info["code"] if isinstance(info["code"], int) and 400 <= info["code"] < 600 else 500
        if not inspect.isgenerator(out):
            out["total_yen"] = SPENT["yen"]   # このアプリで使った料金の合計（ページの上に出す）
            # エラーでも 200 で返す（Colab の接続がエラーの番号を見て中身を差しかえても、原因の見立てが届くように）。番号は記録と JSON に入れる
            self.reply(200, json.dumps(out, ensure_ascii=False).encode("utf-8"), "application/json; charset=utf-8")
            return self.note(code, t0)
        # 少しずつ返す：1行に1つの JSON。t は頼まれてからのミリ秒（ブラウザはこの間隔どおりに見せる）
        self.send_response(200)
        self.send_header("Content-Type", "application/x-ndjson; charset=utf-8")
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        code = 200
        try:
            for ev in out:
                ev["t"], ev["total_yen"] = round((time.time() - t0) * 1000), SPENT["yen"]
                self.wfile.write((json.dumps(ev, ensure_ascii=False) + "\n").encode("utf-8"))
                self.wfile.flush()
        except Exception as e:
            info = note_error(name, e)
            code = info["code"] or 500
            try:
                self.wfile.write((json.dumps({"type": "error", "error": info, "total_yen": SPENT["yen"]},
                                             ensure_ascii=False) + "\n").encode("utf-8"))
            except OSError:
                pass   # ブラウザが先に閉じた
        self.note(code, t0)


print("準備ができました。次は 0-2 で APIキーを入れます")

## 0-2 APIキーを入れる
実行すると入力欄が出るので、APIキーを貼り付けて Enter を押します（前回と同じ `getpass` なので、画面には `••••` と表示されます）。

In [ ]:
import os
import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEYを入力してください: ")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("APIキーを設定しました")

## 0-3 アプリを起動する
▶ を押すと、サーバーと APIキーの状態と、オレンジのボタンが出ます。押すと **新しいタブにアプリ** が開きます。
- アプリはこの Colab の中で動いています。Colab のタブは閉じないでください
- 開かないときは、ボタンの下の「開かないとき」を開きます。それでも分からなければ **0-4** を実行します

In [ ]:
# @title アプリを起動する
from google.colab.output import eval_js

try:
    app_server.shutdown()
    app_server.server_close()
except NameError:
    pass
for PORT in range(8000, 8010):   # 前のサーバーが残っていてその番号が使えないときは、次の番号にする
    try:
        app_server = ThreadingHTTPServer(("0.0.0.0", PORT), AppHandler)
        break
    except OSError:
        continue
else:
    raise RuntimeError("使える番号がありません。「ランタイム → セッションを再起動」して 0-1 から実行してください")
threading.Thread(target=app_server.serve_forever, daemon=True, name="app-server").start()
LOG.update(port=PORT, started=now(), requests=[])

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
display(HTML(start_panel(url, PORT)))

## 0-3b 新しいタブで開かないとき
このセルの中にアプリを開きます（0-3 を実行したあとに使う）。カメラやマイクが使えないことがあるので、まずは 0-3 のボタンを試します。

In [ ]:
# @title このセルの中でアプリを開く
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=900)

## 0-4 困ったとき（診断）
アプリが開かない・エラーが出るときに実行します。サーバー・APIキー・モデル（料金はかからない確かめ方）・ブラウザから届いたアクセス・最近のエラーを調べて、**いちばん怪しいところ** を出します。講師に見せるときも、この結果を見せます。

In [ ]:
# @title 調べる
display(HTML(diagnose()))

## アプリでやること

| ページ | やること |
|---|---|
| 1 画像分類 | 物を撮る → マスの数・ノイズ・回転・明るさを変える → 「調べる」で AIが見ている場所を見る |
| 2 手と体の検出 | 指を動かす → しきい値を変える → AIが出した63個の数字と、決まりの棒を見る |
| 3 画像を読み取る | 写真を撮る → 質問と detail を選ぶ →「読み取る」。low と high のマスの数と料金の内訳を見る →「答えの形：JSON」で物と位置を受け取り、枠を描く |
| 4 音の波形 | 「あいうえお」で模様を見る → 録音して測る回数を下げる → 模様をクリックして1列の作り方を見る |
| 5 文字起こし | 録音して AIに送る形を見る → A：ノイズと測る回数を変える ／ B：ヒントを足す。文字が少しずつ届く |
| 6 声で話すChatbot | キャラクターと声を決める →「話す」で話しかける → 3つのAIのリレーの動き・毎回送っているメッセージの束・1回ごとの時間とお金を見る |
| 7 意見メモ | グループでもらった意見を3つの欄に書く →「AIに聞く」で次にやることの候補を出す →「💾 PCに保存」で持ち帰る |

## 終わったら
- 講座で配られた APIキーは、講座の中だけで使います。ノートブックやメモに書き写したり、人に教えたりしないでください
- 1・2・4 は、配られた体験ページ（HTML）をブラウザで開けば、家でも APIキーなしで試せます